# L'optimisation par SMT (Z3) : le plus petit cover

**Search / Part2-CSP** — le carnet **CSP-10**.

Tous les carnets de la série CSP (`CSP-1`..`CSP-9`) s'appuient sur **CP-SAT** (OR-Tools) en Python ; **Choco** n'en est que le pont **.NET** (via IKVM) sur les binômes. `CSP-5 — Optimization` y montrait déjà l'optimisation.

Ce carnet ajoute l'autre **organe** de la boîte à outils : le solveur **SMT**, **Z3** (le solveur SMT/SAT de référence, ici sa binding Python `z3-solver`). On attaque la **même classe de problème d'optimisation** — *trouver le plus petit ensemble qui couvre tout* — deux façons :

1. une **baseline brute de force** (on énumère les `2^k` sous-ensembles) ;
2. **Z3 en mode optimiseur** (`Optimize` + `minimize`), qui renvoie l'optimum **et le prouve minimal**.

L'objectif pédagogique (un vrai outil SOTA sur un problème non-trivial) est de rendre **visible** la capacité distinctive du solveur : sur une instance où la force brute explose, Z3 trouve le minimum et **certifie** qu'il est optimal, en une fraction de seconde.


In [1]:
import z3
import time
from itertools import combinations

print("z3-solver", z3.get_version_string())


z3-solver 4.15.4


## 1. Le problème : le *minimum set cover*

On a une grille de **points à couvrir** (les *cells*). On a une liste de **positions candidates** où l'on peut placer un **capteur** ; chaque capteur couvre un **voisinage** (ici un carré `3x3`, distance de Chebyshev `<= 1`, rogné sur la grille).

On cherche la **plus petite** liste de capteurs qui couvre **tous** les points. C'est le problème du **couvre-ensemble minimum** (*set cover*), **NP-difficile** en général.

La nuance qui compte ici : on ne demande pas seulement « existe-t-il un cover de taille `<= k` ? » (une question de *décision*, que la force brute trancherait en énumérant). On demande **le minimum** — et surtout une **preuve** qu'on ne peut pas faire mieux. C'est exactement ce que l'optimiseur d'un solveur SMT est fait pour fournir.

In [2]:
def make_instance(n, drop_last_col=False):
    """Grille n x n. Cells = tous les points. Candidates = un sous-ensemble
    deterministe des positions (on peut retirer la derniere colonne pour
    rendre le pool de candidats plus petit que le nombre de points).
    covers[i] = ensemble des points couverts par le capteur i (carre 3x3, Chebyshev)."""
    cells = [(r, c) for r in range(n) for c in range(n)]
    cand  = [p for p in cells if not (drop_last_col and p[1] == n - 1)]
    covers = []
    for (r, c) in cand:
        covers.append(frozenset(
            (r2, c2) for (r2, c2) in cells
            if abs(r - r2) <= 1 and abs(c - c2) <= 1
        ))
    return cells, cand, covers

# Petite instance : 4x4, 16 points, 16 positions candidates  ->  2^16 sous-ensembles
small_cells, small_cand, small_covers = make_instance(4)
# Grande instance : 8x8, 64 points, 64 positions candidates  ->  2^64 sous-ensembles
large_cells, large_cand, large_covers = make_instance(8)

print(f"petite : {len(small_cells)} points, {len(small_cand)} candidats, 2^{len(small_cand)} = {2**len(small_cand):,} sous-ensembles")
print(f"grande : {len(large_cells)} points, {len(large_cand)} candidats, 2^{len(large_cand)} = {2**len(large_cand):,} sous-ensembles")


petite : 16 points, 16 candidats, 2^16 = 65,536 sous-ensembles
grande : 64 points, 64 candidats, 2^64 = 18,446,744,073,709,551,616 sous-ensembles


### Lecture du resultat

La **petite** instance offre `2^16 = 65 536` sous-ensembles — énumérables en une fraction de seconde. La **grande** en offre `2^64 \approx 1,8 \times 10^{19}` sous-ensembles — un ordre de grandeur que la force brute ne traversera **pas** dans une fenêtre raisonnable. C'est l'écart que ce carnet va faire **voir**.

## 2. La baseline : la force brute

L'approche naïve : tester les sous-ensembles **par taille croissante** (1 capteur, puis 2, puis 3, ...) et renvoyer le **premier** qui couvre tout. Le premier cover complet trouvé *est* le minimum, car on monte en taille. Coût : jusqu'à `2^k` tests, chacun d'un coût `O(k)` pour fusionner les voisinages.

In [3]:
def brute_min_cover(covers, cells):
    """Minimum set cover par enumeration, par taille croissante.
    Renvoie (taille_min, liste_de_positions) — ou None si rien ne couvre."""
    allc = frozenset(cells)
    k = len(covers)
    for size in range(1, k + 1):
        for combo in combinations(range(k), size):
            union = frozenset()
            for i in combo:
                union |= covers[i]
            if union == allc:
                return size, list(combo)
    return None

t0 = time.perf_counter()
res = brute_min_cover(small_covers, small_cells)
dt = time.perf_counter() - t0
print(f"force brute (petite) : minimum = {res[0]} capteurs, temps = {dt*1000:.1f} ms")
print(f"  ex. de positions : {res[1]}")


force brute (petite) : minimum = 4 capteurs, temps = 1.3 ms
  ex. de positions : [0, 2, 8, 10]


### Lecture du resultat

Sur la petite instance, la force brute trouve le minimum **et** un cover concret, en quelques millisecondes — `2^16` est petit. Rien de spectaculaire : c'est le **point de départ** de la comparaison, pas le héros de l'histoire.

## 3. Z3 en mode **optimiseur**

On déclare une variable booléenne `x[i]` par position candidate (« le capteur `i` est-il placé ? »). On ajoute, pour **chaque point**, la contrainte « au moins un capteur qui le couvre est placé ». L'**objectif** est `minimize(Σ x[i])` — le nombre de capteurs.

Z3 n'est pas qu'un solveur de satisfaction : `z3.Optimize` fait un **branch-and-bound** sur l'objectif. Quand il renvoie le statut `optimal`, il ne s'est pas contenté de *trouver* un cover minimal : il a **prouvé** qu'aucun cover plus petit n'existe. C'est la certification que la force brute ne fournit pas (elle s'arrête au premier cover trouvé).

In [4]:
def z3_min_cover(covers, cells):
    """Minimum set cover via Z3 Optimize.
    Renvoie (statut, objectif_minimal, liste_de_positions)."""
    s = z3.Optimize()
    x = [z3.Bool(f"x{i}") for i in range(len(covers))]
    for (r, c) in cells:
        covering = [x[i] for i in range(len(covers)) if (r, c) in covers[i]]
        s.add(z3.Or(*covering))                       # chaque point est couvert
    s.minimize(z3.Sum([z3.If(xi, 1, 0) for xi in x])) # minimum de capteurs
    chk = s.check()
    if chk == z3.sat:   # pour un Optimize apres minimize, sat = optimum certifie
        m = s.model()
        placed = [i for i in range(len(covers))
                  if m.eval(x[i], model_completion=True) == z3.BoolVal(True)]
        return "optimal", len(placed), placed
    return str(chk), None, None

t0 = time.perf_counter()
stat, opt, placed = z3_min_cover(small_covers, small_cells)
dt = time.perf_counter() - t0
print(f"Z3 (petite) : statut = {stat}, minimum = {opt} capteurs, temps = {dt*1000:.1f} ms")
print(f"  positions : {placed}")
print(f"  -> coincide avec la force brute ?  {opt == res[0]}")


Z3 (petite) : statut = optimal, minimum = 4 capteurs, temps = 101.9 ms
  positions : [3, 5, 12, 14]
  -> coincide avec la force brute ?  True


### Lecture du resultat

Sur la petite instance, **Z3 et la force brute tombent d'accord** sur le minimum — c'est le **témoin de correction** : on sait que le solveur calcule la bonne réponse, pas juste *une* réponse. Et il l'obtient avec un statut `optimal`, c'est-à-dire **certifié**.

## 4. Le vrai enjeu : faire **voir** la capacité du solveur

On passe à la **grande** instance (`2^64 \approx 1,8 \times 10^{19}` sous-ensembles). La force brute est **inexploitable** à cette échelle. Z3, lui, résout et **certifie** l'optimum en une fraction de seconde. L'écart n'est pas « Z3 est un peu plus rapide » : c'est « Z3 **répond**, la force brute **ne peut pas** ».

In [5]:
# La force brute : on ne la lance PAS sur 2^64. On affiche seulement ce qu'elle devrait faire.
print(f"force brute (grande) : devrait énumérer jusqu'a 2^{len(large_cand)} = {2**len(large_cand):,} sous-ensembles -> inexploitable")

# Z3 : résolution + certification de l'optimum
t0 = time.perf_counter()
stat, opt, placed = z3_min_cover(large_covers, large_cells)
dt = time.perf_counter() - t0
print(f"Z3 (grande)    : statut = {stat}, minimum = {opt} capteurs, temps = {dt*1000:.1f} ms")
print(f"  {opt} capteurs placés, optimum CERTIFIÉ (statut optimal) sur {len(large_cells)} points / {len(large_cand)} candidats")


force brute (grande) : devrait énumérer jusqu'a 2^64 = 18,446,744,073,709,551,616 sous-ensembles -> inexploitable


Z3 (grande)    : statut = optimal, minimum = 9 capteurs, temps = 71.2 ms
  9 capteurs placés, optimum CERTIFIÉ (statut optimal) sur 64 points / 64 candidats


### Lecture du resultat

C'est le cœur du carnet : sur la grande instance, **Z3 renvoie le minimum et le prouve** (`statut = optimal`) en moins d'une seconde, là où la force brute serait perdue dans `1,8 \times 10^{19}` sous-ensembles. La capacité distinctive du solveur SMT — *propagation + branch-and-bound + preuve d'optimalité* — est ici **mesurable**, pas illustrée.

## 5. Ce que le solveur fait (en une phrase)

Z3 ne teste pas les `2^k` combinaisons une à une : il **propage** les contraintes (chaque point non couvert force le placement d'un capteur qui le couvre), **branche** sur les choix restants, et le **branch-and-bound** de l'optimiseur élimine les branches qui ne peuvent pas battre la meilleure solution courante. C'est cette élimination *avant* d'explorer qui rend l'optimum atteignable — et certifiable — là où l'énumération brute s'effondre.

## 6. Exercices

In [6]:
# Exercice 1
# TODO etudiant : ecrire la fonction qui, donnee une instance (covers, cells),
# construit l'objet z3.Optimize avec les contraintes de couverture et
# l'objectif de minimisation, puis renvoie (statut, minimum, positions).
# # Indice : c'est exactement z3_min_cover ci-dessus ; en ecris la version
# #         qui prend (covers, cells) en parametre et renvoie le 3-uplet.
def z3_min_cover_etudiant(covers, cells):
    # TODO etudiant
    return None  # TODO etudiant : renvoyer (statut, minimum, positions)


In [7]:
# Exercice 2
# TODO etudiant : ecrire la baseline brute de force qui renvoie le minimum
# set cover (par taille croissante) d'une instance donnee.
# # Indice : iterer la taille de 1 a k, et pour chaque taille tester les
# #         combinations(k, size) ; renvoyer la premiere qui couvre tout.
def brute_min_cover_etudiant(covers, cells):
    # TODO etudiant
    return None  # TODO etudiant : renvoyer (taille_min, positions)


In [8]:
# Exercice 3
# TODO etudiant : le solveur donne le minimum k*. Trouver le DEUXIEME
# minimum : ajouter la contrainte qui EXCLUT tout cover de taille k*
# (c-a-d imposer au moins k*+1 capteurs) et re-resoudre.
# # Indice : re-partir de z3_min_cover, ajouter s.add(z3.Sum([z3.If(xi,1,0)
# #         for xi in x]) >= k_star + 1), et relire l'objectif.
def second_min_cover_etudiant(covers, cells, k_star):
    # TODO etudiant
    return None  # TODO etudiant : renvoyer le deuxieme minimum (k* + 1 typiquement)


## En synthèse

La série CSP s'appuyait sur **CP-SAT** (OR-Tools, Python) et **Choco** (.NET). Ce carnet CSP-10 ajoute l'organe **SMT** : **Z3**, utilisé ici en **optimiseur** sur le *minimum set cover*.

- la **force brute** énumère les `2^k` sous-ensembles : suffisante à `k = 16` (`65 536`), **inexploitable** à `k = 64` (`1,8 \times 10^{19}`) ;
- **Z3 `Optimize`** résout les deux, et surtout **certifie l'optimalité** (statut `optimal`) en une fraction de seconde, y compris sur l'instance que la force brute ne peut pas aborder.

C'est la leçon de la série : le bon outil ne se choisit pas au nom, mais à sa **capacité distinctive** — et celle-ci, on la **mesure** plutôt qu'on ne la présume.